# 2019 年旧表单 SOXX 系列身份校验

按预先冻结的 `identity_repair_plan.json`，只重读两份已归档 SEC 索引原文；不再发请求。

In [1]:
import json,hashlib
from pathlib import Path
from bs4 import BeautifulSoup
root=Path.cwd();card=root/'research/stage1_soxx_2019_transition_index_20261023';plan=json.loads((card/'identity_repair_plan.json').read_text());parent=json.loads((card/'metadata_result.json').read_text());transport=json.loads((card/'transport_result.json').read_text());receipts=json.loads((card/'capture_receipts.json').read_text())
assert plan['frozen'] and transport['nas_remote_and_scoped_restore_verified'] and len(receipts)==2
raw=root/'data/snapshots/stage1_soxx_2019_transition_index_20261023';results=[]
for receipt in receipts:
 path=raw/(receipt['accession']+'.html');assert hashlib.sha256(path.read_bytes()).hexdigest()==receipt['raw_sha256']
 soup=BeautifulSoup(path.read_bytes(),'html.parser');div=soup.select_one('#seriesDiv');assert div
 rows=[[cell.get_text(' ',strip=True) for cell in tr.find_all('td',recursive=False)] for tr in div.select('tr')]
 series_positions=[i for i,row in enumerate(rows) if row and row[0]=='Series S000004354']
 class_positions=[i for i,row in enumerate(rows) if row and row[0]=='Class/Contract C000012084']
 passed=len(series_positions)==len(class_positions)==1 and class_positions[0]==series_positions[0]+1 and 'SOXX' in rows[class_positions[0]] and 'iShares PHLX Semiconductor ETF' in rows[series_positions[0]] and 'iShares PHLX Semiconductor ETF' in rows[class_positions[0]]
 results.append({'accession':receipt['accession'],'series_row':rows[series_positions[0]] if len(series_positions)==1 else None,'class_row':rows[class_positions[0]] if len(class_positions)==1 else None,'adjacent_rows':len(series_positions)==len(class_positions)==1 and class_positions[0]==series_positions[0]+1,'identity_passed':passed,'raw_sha256':receipt['raw_sha256']})
result={'status':'two_2019_transition_index_series_class_row_identity_passed' if len(results)==2 and all(x['identity_passed'] for x in results) else 'transition_index_identity_stop','repair_plan_sha256':hashlib.sha256((card/'identity_repair_plan.json').read_bytes()).hexdigest(),'source_archive_sha256':transport['nas_archive_sha256'],'NAS_scoped_restore_verified':True,'rows':results,'original_holdings_values_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'identity_result.json').write_text(json.dumps(result,indent=2)+'\n')
print({'status':result['status'],'rows':results})

{'status': 'two_2019_transition_index_series_class_row_identity_passed', 'rows': [{'accession': '0001193125-19-059323', 'series_row': ['Series S000004354', '', 'iShares PHLX Semiconductor ETF'], 'class_row': ['Class/Contract C000012084', '', 'iShares PHLX Semiconductor ETF', 'SOXX'], 'adjacent_rows': True, 'identity_passed': True, 'raw_sha256': '1ab01adbc91d1b56894956039d3a03828e31605382231afe731be20b1c37f2a5'}, {'accession': '0001193125-19-167652', 'series_row': ['Series S000004354', '', 'iShares PHLX Semiconductor ETF'], 'class_row': ['Class/Contract C000012084', '', 'iShares PHLX Semiconductor ETF', 'SOXX'], 'adjacent_rows': True, 'identity_passed': True, 'raw_sha256': '9fa94a9cfcdba6cfcc1c936d4abf1230f9441e30007bb75c5e5dac88466a18c0'}]}


## 中文结论

逐行确认系列与份额类别相邻后，才将索引身份视为通过。原始持仓正文、Stage1 特征、收益和账户仍未验证。